# 手撕 Contrastive Search

## 背景
对比搜索（Contrastive Search）兼顾质量与多样性。
每步选 token 时，不仅看当前概率，还惩罚与历史 token 的表示相似度。
score = (1-α) * p(v) - α * max_k cos(h(v), h(y_k))

## 考察点
- 对比惩罚的直觉（避免重复，保持连贯）
- α 的调节（0=贪心，1=最大多样性）
- 与 beam search / sampling 的对比

In [ ]:
import torch
import torch.nn.functional as F

def contrastive_search(logits, hidden_states, alpha=0.6, k=4):
    # logits: (vocab_size,) 当前步
    # hidden_states: (seq_len, hidden_dim) 已生成 token 的隐表示
    # alpha: 惩罚系数, k: top-k 候选
    # Step 1: 取 top-k 候选
    topk_probs, topk_ids = F.softmax(logits, dim=-1).topk(k)
    # Step 2: 对每个候选 v，计算惩罚项
    # 假设候选的 hidden 表示用 logits 的 embedding 近似
    # 这里用 one-hot 近似 hidden（实际应用中用模型 hidden）
    candidate_hidden = F.one_hot(topk_ids, num_classes=logits.size(-1)).float()
    past_hidden = F.one_hot(torch.arange(hidden_states.size(0)), num_classes=logits.size(-1)).float()
    # 计算最大余弦相似度
    cos_sim = F.cosine_similarity(
        candidate_hidden.unsqueeze(1),  # (k, 1, vocab)
        past_hidden.unsqueeze(0),       # (1, seq, vocab)
        dim=-1
    ).max(dim=1).values  # (k,)
    # Step 3: 综合得分
    scores = (1 - alpha) * topk_probs - alpha * cos_sim
    best_idx = scores.argmax()
    return topk_ids[best_idx].item(), topk_probs[best_idx].item()

In [ ]:
# 验证对比搜索
torch.manual_seed(42)
vocab_size = 100
logits = torch.randn(vocab_size)
hidden_states = torch.randn(10, vocab_size)  # 10 个已生成 token
# α=0 时退化为贪心
token_greedy, _ = contrastive_search(logits, hidden_states, alpha=0.0)
greedy_token = logits.argmax().item()
assert token_greedy == greedy_token, "α=0 应退化为贪心"
# α=0.6 时应可能选不同 token
token_cs, prob = contrastive_search(logits, hidden_states, alpha=0.6)
print(f"贪心 token: {greedy_token}")
print(f"对比搜索 token: {token_cs} (prob={prob:.4f})")
print("✅ Contrastive Search 验证通过")